# Reproducing Lefebvre et al. (2008), Figure 9, from CRCNS ret-1

Paper. Lefebvre JL, Zhang Y, Meister M, Wang X, Sanes JR (2008). Gamma-Protocadherins regulate
neuronal survival but are dispensable for circuit formation in retina. *Development*
135:4141-4151. [doi:10.1242/dev.027912](https://doi.org/10.1242/dev.027912). The claims file was
written from the NIH author manuscript in PubMed Central (PMC2644426), with its Methods, Results,
Discussion and figure legends; the supplementary figures were not available to it. The paper
links no analysis code.

Data set. CRCNS ret-1, "Multi-electrode recordings from retinal ganglion cells" (Zhang, Asari
and Meister 2014), [doi:10.6080/K0RF5RZT](https://doi.org/10.6080/K0RF5RZT), read from the public
S3 bucket `crcnsarchive` under `ret-1/`. CRCNS data sets are not versioned, so the version is
recorded as `unversioned`. There is one form of the data set to choose: the paper (2008) predates
it and cites none, and the bucket holds one zip file, `crcns_ret-1.zip` (71,871,914 bytes), whose
MD5 in the data set's `checksums.md5` is `4b9390ddc51c72d6f66563cb02e48fe8`; the data check
below streams the zip and confirms that sum, and lists the CRC-32 of every member read, so that a
later change is detectable. The description is version 0.8 (2 September 2014) and the files in
the zip are dated 2014. The zip is opened in place over range requests; nothing is written to disk.

What the data set holds. Spike times of 325 sorted units from 16 isolated retinas (5 dates in
2008), each file labelled with a genotype, and their responses to binary white noise only:
flickering vertical and horizontal bars and checkerboards, drawn from the RAN1 generator whose
bits are in the zip. It holds none of the histology of Figures 1 to 8, and none of the flashing
spots, flashes, moving bars or gratings of Figure 9A to 9C.

Outside inputs. None.

| Claim | Tested here |
|---|---|
| fig1-broad-expression, fig2-postnatal-localization, fig3-synaptic-localization, s1-chx10-recombination, fig4h-inl-ipl-thinner, fig4-alleles-equivalent, fig4i-cell-type-loss, fig5-postnatal-onset, fig5-increased-apoptosis, fig6f-amacrine-loss-central, fig6f-neighbours, fig7-laminar-targeting, fig8ad-bax-rescue, fig8el-synapse-density, fig8mn-bax-laminar | Blocked: histology, not in the data set |
| fig9a-response-variety, fig9c-on-off-index | Blocked: no flashing-spot recordings |
| fig9-rod-cone | Blocked: no flashes of two intensities |
| fig9b-direction-selective | Blocked: no moving gratings or bars |
| fig9-cell-counts | Tested |
| fig9d-rf-size-similar | Tested |
| s3-receptive-field-surrounds | Tested |
| fig9e-slower-time-course | Tested |
| fig9f-lower-firing-rate | Tested |
| fig9g-threshold-unaltered | Tested |
| fig9h-gain-reduced | Tested |

Which bar recording. The claims file names bars of 8.3 or 16.6 um but not their orientation,
nor what to do when a retina has more than one such recording. This was asked
(`questions/crcns-ret-1.yaml`), and every claim on the bars is tested under three readings: (A)
the first vertical-bar recording of each retina, (B) all its vertical-bar recordings pooled, (C)
its horizontal-bar recording where it has one, otherwise as A. Each claim's outcome is the one all
readings give.

In [ ]:
import hashlib
import io
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pypdf
import requests
from scipy import stats

import ret1

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
z = ret1.open_zip()
retinas = ret1.load_retinas(z)
by_name = {R["retina"]: R for R in retinas}
nbits = max((r["x"] // r["dx"]) * (r["y"] // r["dy"]) * r["nframes"] for R in retinas for r in R["recordings"])
bits = ret1.ran1_bits(z, nbits)
print(f"CRCNS {ret1.DATASET_ID} ({ret1.VERSION}, doi:{ret1.DATASET_DOI}): {len(retinas)} retina files and "
      f"{nbits:,} stimulus bits loaded in {time.time() - started:.0f} s")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
# Integrity of the zip: its MD5, streamed in memory, against the data set's checksums.md5.
sums = dict(line.split()[::-1] for line in ret1.read_text("checksums.md5").splitlines() if line.strip())
md5 = hashlib.md5()
with requests.get(ret1.BASE_URL + ret1.ZIP_NAME, stream=True, timeout=120) as r:
    for chunk in r.iter_content(2**20):
        md5.update(chunk)
print("checksums.md5:", sums)
print("MD5 of", ret1.ZIP_NAME, "as streamed:", md5.hexdigest(), "; matches checksums.md5:",
      md5.hexdigest() == sums[ret1.ZIP_NAME] == ret1.ZIP_MD5)
print("\nMembers read (size in bytes and CRC-32 from the zip directory):")
for info in z.infolist():
    if info.filename.startswith(ret1.PREFIX) and not info.is_dir():
        print(f"  {info.filename:40s} {info.file_size:>10,d}  {info.CRC:08x}")

In [ ]:
# The description's table of retinas against the files.
pdf = pypdf.PdfReader(io.BytesIO(requests.get(ret1.BASE_URL + "docs/crcns_ret-1_data_description.pdf", timeout=60).content))
text = "\n".join(p.extract_text() for p in pdf.pages)
described = {m[0]: (int(m[1]), m[2].strip()) for m in re.findall(r"(\d{8}_R\d)\s*\(\s*(\d+) cells\):\s*([^\n]+)", text)}
info = pd.DataFrame([{
    "retina": R["retina"], "label": R["animal"], "group": ret1.group(R["animal"]), "cells": R["ncell"],
    "described_cells": described.get(R["retina"], (None, None))[0],
    "described_label": described.get(R["retina"], (None, None))[1],
    "recordings": len(R["recordings"]),
    "stimuli": ", ".join(f"{ret1.kind(r)} {ret1.bar_width_um(r) if ret1.kind(r) != 'checker' else r['dx'] * r['pixelsize']:.1f} um "
                         f"{r['frame'] * 1000:.2f} ms x {r['nframes']}" for r in R["recordings"]),
} for R in retinas])
print(info.to_string(index=False))
print("\nRetinas in the description:", len(described), "; files:", len(retinas))
print("Cell counts and labels agree with the description for every retina:",
      bool((info.cells == info.described_cells).all() and (info.label == info.described_label).all()))
print("\nCells by label:"); print(info.groupby("label").cells.agg(["count", "sum"]).rename(columns={"count": "retinas", "sum": "cells"}).to_string())
print("\nCells by group used here (mutant = KO without Bax, control = WT):")
print(info.groupby("group").cells.agg(["count", "sum"]).rename(columns={"count": "retinas", "sum": "cells"}).to_string())
print("\nStimulus types in the data set:", sorted({r["type"] for R in retinas for r in R["recordings"]}))
print("Seeds:", sorted({r["seed"] for R in retinas for r in R["recordings"]}), "; pixel sizes (um):",
      sorted({r["pixelsize"] for R in retinas for r in R["recordings"]}))
print("Frame lengths (ms):", sorted({round(r["frame"] * 1000, 3) for R in retinas for r in R["recordings"]}))
print("Experimenter:", sorted({R["experimenter"] for R in retinas}), "; dates:", sorted({R["date"] for R in retinas}))

The 16 files match the description's table in cell counts and genotype labels. Every recording is
binary white noise drawn from RAN1 with seed -10000: vertical bars (full height), horizontal bars
(full width) or checkerboards. There are no flashing spots, flashes of different intensity, moving
bars or drifting gratings, so the stimuli of Figure 9A, 9B and 9C and of the rod and cone test
are not in the data set. The pixel size is 8.3 um as stated, so the bars are 8.3, 16.6 and (one
recording) 41.5 um wide, the widths the Methods give except the last. Frames last 16.67 ms (60 Hz)
for the bars, as the Methods state, and 33.3 ms for some checkerboards; two recordings of 20080702
state 16.665 and 16.669 ms.

The labels give 200 mutant cells in 10 retinas (Chx10-Cre 87, Pax6-Cre 113), 3 cells from one
Bax -/- mutant retina, and 122 control cells in 5 retinas, 95 of them labelled "chx10 het". The
paper's Figure 9 has 115 mutant and 143 control cells, and names its controls Pax6-Cre;
Pcdh-gamma +/f-del or Pcdh-gamma f-del/f-del without Cre; it names no Chx10-Cre heterozygous
controls. The files do not say which part of the Pax6-Cre retinas was recorded (the paper used
their peripheral region only), nor the age of the mice, nor the allele.

In [ ]:
# ran1.bin against the RAN1 generator of ran1.m, and the bit order.
check = ret1.ran1(-10000, 20000) >= 0.5
with z.open(ret1.PREFIX + "ran1.bin") as f:
    raw = np.frombuffer(f.read(2500), dtype=np.uint8)
for order in ("little", "big"):
    agree = (np.unpackbits(raw, bitorder=order)[:20000].astype(bool) == check).mean()
    print(f"first 20000 bits, {order}-endian bit order: {agree:.4f} agree with RAN1(seed -10000) >= 0.5")
print("ran1.bin holds", z.getinfo(ret1.PREFIX + "ran1.bin").file_size * 8, "bits; the longest stimulus needs", nbits)

In [ ]:
# Spike times: units, order, and range against the stimulus window.
rows = []
for R in retinas:
    for rec in R["recordings"]:
        end = rec["onset"] + rec["nframes"] * rec["frame"]
        parts = ret1.split_trains(rec)
        tails = [tl for _, tl in parts if len(tl)]
        allt = np.concatenate([t for t in rec["spikes"] if len(t)])
        rows.append({"retina": R["retina"], "rec": rec["index"], "kind": ret1.kind(rec), "onset_s": round(rec["onset"], 2),
                     "stim_end_s": round(end, 1), "first_spike_s": round(allt.min(), 3), "last_spike_s": round(allt.max(), 1),
                     "spikes_after_end": int((allt > end).sum()),
                     "unsorted_trains": len(tails), "head_ends_s": round(max((h.max() for h, tl in parts if len(tl)), default=np.nan), 1),
                     "tail_span_s": (f"{min(t.min() for t in tails):.1f}-{max(t.max() for t in tails):.1f}" if tails else ""),
                     "tail_spikes": int(sum(len(t) for t in tails))})
spk = pd.DataFrame(rows)
print(spk.to_string(index=False))
print("\nRecordings with out-of-order spike times:", int((spk.unsorted_trains > 0).sum()), "of", len(spk),
      "; trains affected:", int(spk.unsorted_trains.sum()), "; spikes in the out-of-order segments:", int(spk.tail_spikes.sum()))

In [ ]:
# Where do the out-of-order segments belong? STA of the tail spikes alone, at their stored times and
# moved by 720 s (to 1440-1500 s, the part of the recording that has no spikes otherwise).
def peak_ratios(rec, trains, nlag=15):
    r = dict(rec); r["spikes"] = trains
    S, C = ret1.stimulus(r, bits), ret1.counts(r)
    h, n = ret1.sta(S, C, nlag)
    hn, _ = ret1.sta(S, np.roll(C, C.shape[0] // 2, axis=0), nlag)
    with np.errstate(invalid="ignore"):
        return np.abs(h).reshape(len(n), -1).max(1) / np.abs(hn).reshape(len(n), -1).max(1), n

tail_rows = []
for R in retinas:
    rec = R["recordings"][0]
    parts = ret1.split_trains(rec)
    if ret1.kind(rec) == "checker" or not any(len(tl) for _, tl in parts):
        continue
    stored, n = peak_ratios(rec, [tl for _, tl in parts])
    moved, _ = peak_ratios(rec, [tl + 720.0 for _, tl in parts])
    ok = n >= 50
    tail_rows.append({"retina": R["retina"], "rec": rec["index"], "cells_with_50_tail_spikes": int(ok.sum()),
                      "median_ratio_stored": round(float(np.median(stored[ok])), 2), "median_ratio_moved_720s": round(float(np.median(moved[ok])), 2),
                      "cells_higher_when_moved": int((moved[ok] > stored[ok]).sum())})
print(pd.DataFrame(tail_rows).to_string(index=False))
print("\nTrains without an out-of-order segment in the same recordings: spikes in 1440-1500 s and in 720-780 s")
for R in retinas:
    for rec in R["recordings"]:
        parts = ret1.split_trains(rec)
        if any(len(tl) for _, tl in parts) and rec["nframes"] * rec["frame"] > 1450:
            for c, (h, tl) in enumerate(parts):
                if not len(tl):
                    print(f"  {R['retina']} rec {rec['index']} unit {c}: {int((h > 1440).sum())} and {int(((h > 720) & (h < 780)).sum())}")

Spike times are in seconds from the start of each recording and begin before the stimulus onset
(about 1 s), as the example script assumes. In the 8 recordings of 20080628_R4 and R6 and of 20080702_R1 and R4
(vertical bars and checkerboards, each of 1500 s), almost every train is out of order: it runs from
0 to 1440 s and then holds a second segment of spikes between 720 and 780 s, though the stimulus
runs to 1500 s. The few trains in these recordings that are in order have spikes between 1440 and
1500 s, about as many as between 720 and 780 s (list above). Moved by
720 s, to 1440 to 1500 s, these segments give higher STA peaks over the null than where they are
stored, in most cells of the larger retinas (table above). The segments are
consistent with the last minute of each recording stored with the wrong offset, but this is
not proven. The main analysis uses the spike times as deposited; a variant moves the segments.
(One checkerboard recording, 20080628_R5, has 1 or 2 spikes near 2 s out of order in several
cells; it is not used.)

In [ ]:
# Units sorted twice: pairs with more than half of the smaller train's spikes within 0.5 ms of the other's.
dups = []
for R in retinas:
    for rec in R["recordings"]:
        for a, b, f in ret1.duplicate_pairs(rec):
            dups.append({"retina": R["retina"], "rec": rec["index"], "group": ret1.group(R["animal"]), "unit_a": a, "unit_b": b, "coincident": round(f, 3)})
dups = pd.DataFrame(dups)
pairs = dups.groupby(["retina", "group", "unit_a", "unit_b"]).coincident.agg(["min", "max", "count"]).reset_index()
print(pairs.to_string(index=False))
print("\nDistinct pairs:", len(pairs), "; by group:", pairs.groupby("group").size().to_dict())
# Each pair in the first vertical-bar recording, and the second unit dropped in the variant
first_vb = {R["retina"]: ret1.bar_recordings(R, "A")[0]["index"] for R in retinas}
drop = set()
for _, p in dups.iterrows():
    if p.rec == first_vb[p.retina]:
        drop.add((p.retina, int(p.unit_b)))
print("Units dropped in the variant without duplicates (second of each pair in the vertical-bar recording):", len(drop))
seen = {}
for R in retinas:
    for rec in R["recordings"]:
        for c, t in enumerate(rec["spikes"]):
            if len(t):
                seen.setdefault(hashlib.md5(np.sort(t).round(4).tobytes()).hexdigest(), []).append((R["retina"], rec["index"], c))
print("Spike trains that occur more than once, across all files, recordings and units:", sum(len(v) > 1 for v in seen.values()))

No recording is deposited twice: no spike train repeats across files or units. Within recordings,
the pairs above share 70 to 100% of their spikes within 0.5 ms in every recording of the retina,
so each pair is most likely one neuron sorted twice. All units are kept in the main analysis, as
the claims count all units; a variant drops the second unit of each pair.

In [ ]:
# Clock: does the stimulus, rebuilt from its onset and frame length, line up with the spikes? Per
# recording, the STA peak over that of a null STA (spike counts circularly shifted by half the
# recording), and the lag of the STA peak. A response aligned to the stimulus gives ratios well
# above 1 at a consistent lag; misaligned timing gives ratios near 1.
clock = []
for R in retinas:
    for rec in R["recordings"]:
        if ret1.kind(rec) == "checker" and R["retina"] not in ("20080702_R4", "20080702_R1", "20080606_R1", "20080606_R2", "20080606_R3"):
            continue
        ratio, n = peak_ratios(rec, rec["spikes"], nlag=ret1.STA_FRAMES)
        S, C = ret1.stimulus(rec, bits), ret1.counts(rec)
        h, _ = ret1.sta(S, C)
        lag_ms = np.array([(np.unravel_index(np.nanargmax(np.abs(x)), x.shape)[1] + 0.5) * rec["frame"] * 1000 if n[i] else np.nan
                           for i, x in enumerate(h)])
        clock.append({"retina": R["retina"], "group": ret1.group(R["animal"]), "rec": rec["index"], "kind": ret1.kind(rec),
                      "cells": len(n), "median_ratio": round(float(np.nanmedian(ratio)), 2), "cells_ratio_above_2": int((ratio > 2).sum()),
                      "median_peak_lag_ms": round(float(np.nanmedian(lag_ms)), 0)})
clock = pd.DataFrame(clock)
print(clock.to_string(index=False))

For most bar recordings the STA peaks lie well above the null at lags of about 100 to 140 ms, so
the stimulus as rebuilt shares the clock of the spikes. The vertical bars of 20080702_R4 are the
exception: none of its 12 cells has an STA above the null (median ratio about 1), while the same
cells give clear STAs to the checkerboard of the same retina. Neither shifting the onset by up to 2
s, nor the usual 16.672 ms frame, nor other bar layouts, recovered a response while preparing (not
run here). So the stimulus of that recording cannot be rebuilt from its description, and its 12
mutant cells enter the main analysis with STAs of noise. The weak ratios of the three 20080606
retinas (all mutant) are matched by weak checkerboard STAs, so they are read as weak responses,
not as misalignment; the alignment there is consistent with the data, not shown. A variant
leaves out 20080702_R4.

## Measures

Every per-cell measure, under each reading of which bar recording is used (A, B, C; see
`questions/crcns-ret-1.yaml`), and, under reading A, with the out-of-order spike segments moved by
720 s. The Bax -/- retina (3 cells) is measured but belongs to neither group: the paper did not use
Bax double mutants for Figure 9, and the claims define the groups without them.

In [ ]:
t0 = time.time()
rows = []
for reading in ret1.READINGS:
    for R in retinas:
        for d in ret1.measure_retina(R, ret1.bar_recordings(R, reading), bits):
            rows.append({"reading": reading, **d})
for R in retinas:
    recs = [ret1.with_tail_shift(r, 720.0) for r in ret1.bar_recordings(R, "A")]
    for d in ret1.measure_retina(R, recs, bits):
        rows.append({"reading": "A, segments moved", **d})
cells = pd.DataFrame(rows)
cells["duplicate"] = [(r, c) in drop for r, c in zip(cells.retina, cells.cell)]
print(f"measured in {time.time() - t0:.0f} s")
print(cells.groupby(["reading", "group"]).agg(cells=("cell", "size"), no_spikes=("spikes_sta", lambda s: int((s == 0).sum())),
                                              bar_um=("bar_um", lambda s: sorted(s.unique())), kind=("kind", lambda s: sorted(s.unique()))).to_string())
num = ["rate_hz", "peak_ratio", "fwhm_svd", "fwhm_peak", "ttp_svd_frame", "ttp_peak_frame", "theta_frame_continuous", "gain_frame_continuous"]
print("\nRanges under reading A (min, max):")
print(cells[cells.reading == "A"].groupby("group")[num].agg(["min", "max"]).T.round(3).to_string())

In [ ]:
GROUPS = ("mutant", "control")
CI = ("binomial", "dkw", "bootstrap")


def split(df, col):
    return df[df.group == "mutant"][col].to_numpy(float), df[df.group == "control"][col].to_numpy(float)


def judge(df, cols, rng, label=None):
    """For each measure column and CI reading: medians, KS test, and whether the 95% bands separate."""
    out = []
    for col in cols:
        m, c = split(df, col)
        m, c = m[np.isfinite(m)], c[np.isfinite(c)]
        ks = stats.ks_2samp(m, c)
        for ci in CI:
            r = ret1.compare_bands(m, c, ci, rng)
            out.append({**({"variant": label} if label else {}), "measure": col, "ci": ci, "n_mut": r["n_mutant"], "n_ctl": r["n_control"],
                        "median_mut": np.median(m), "median_ctl": np.median(c), "ks_p": ks.pvalue,
                        "separate": r["separate"], "mutant_larger_pts": r["mutant_right"], "mutant_smaller_pts": r["mutant_left"],
                        "sep_range": (f"{r['range'][0]:.3g} to {r['range'][1]:.3g}" if r["range"] else "")})
    return pd.DataFrame(out)


def by_retina(df, cols):
    return df[df.group.isin(GROUPS)].groupby(["group", "retina"])[cols].agg(["median", "size"]).round(3)


def variants(rng, cols):
    """The same comparison, reading A, on subsets and on the corrected spike times."""
    A = cells[cells.reading == "A"]
    sets = {
        "all units (main)": A,
        "without duplicate units": A[~A.duplicate],
        "without 20080702_R4": A[A.retina != "20080702_R4"],
        "out-of-order segments moved 720 s": cells[cells.reading == "A, segments moved"],
        "mutant Chx10-Cre only": A[(A.group == "control") | A.animal.str.contains("chx10") & (A.group == "mutant")],
        "mutant Pax6-Cre only": A[(A.group == "control") | A.animal.str.contains("pax6") & (A.group == "mutant")],
        "control Pax6-Cre het only": A[(A.group == "mutant") | (A.animal == "WT (pax6 het)")],
        "cells with STA peak ratio above 2": A[A.peak_ratio > 2],
        "16.6 um bars only": A[A.bar_um == 16.6],
    }
    return pd.concat([judge(d[d.group.isin(GROUPS)], cols, rng, label=k) for k, d in sets.items()], ignore_index=True)


def ecdf_plot(ax, df, col, title, xlabel, scale=1.0):
    for g, colour in (("mutant", "tab:red"), ("control", "black")):
        v = np.sort(df[df.group == g][col].dropna().to_numpy() * scale)
        ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=colour, label=f"{g} ({len(v)})")
        lo, hi = ret1.band(v, v, "binomial")
        ax.fill_between(v, lo, hi, step="post", color=colour, alpha=0.15)
    ax.set_title(title, fontsize=9); ax.set_xlabel(xlabel); ax.set_ylabel("cumulative fraction"); ax.legend(fontsize=8)

## fig9-cell-counts

> Figure 9C to 9H report 115 RGCs from mutant retinas and 143 RGCs from control retinas.
> Expected 115 (tolerance 11.5) mutant and 143 (14.3) control cells, counted as the units of the
> data set's mutant and control recordings, all units and units with a flicker response.

In [ ]:
A = cells[cells.reading == "A"]
counts = pd.DataFrame({
    "all units": A.groupby("group").size(),
    "units with spikes in the bar recording": A[A.spikes_sta > 0].groupby("group").size(),
    "STA peak ratio above 1.5": A[A.peak_ratio > 1.5].groupby("group").size(),
    "STA peak ratio above 2": A[A.peak_ratio > 2].groupby("group").size(),
    "all units without duplicates": A[~A.duplicate].groupby("group").size(),
}).fillna(0).astype(int)
counts.loc["mutant + bax"] = counts.loc["mutant"] + counts.loc["bax"]
print(counts.to_string())
print("\nExpected: mutant 115 (103.5 to 126.5), control 143 (128.7 to 157.3)")
print("Mutant cells by Cre line:", A[A.group == "mutant"].animal.value_counts().to_dict())
print("Control cells by label:", A[A.group == "control"].animal.value_counts().to_dict())
print("No count of control units can exceed", int(counts.loc["control", "all units"]), "(all control units), below the lower bound 128.7")

**Result: not reproduced.** The data set's labels give 200 mutant units (87 Chx10-Cre, 113
Pax6-Cre) and 122 control units, against 115 (103.5 to 126.5) and 143 (128.7 to 157.3). Counting
only units whose STA peak is more than 1.5 times the null's gives 126 mutant units, inside the
tolerance, but no count of control units can exceed 122, below the lower bound. So the control
count misses under any reading of "units with a flicker response". The 3 units of the Bax -/-
retina are in neither group.

## fig9d-rf-size-similar

> Receptive field centre size varies greatly among RGCs, but its distribution is similar in
> control and mutant retinas.

FWHM of |b(x)| (um), under the two separations of h and the three band readings, and the three
readings of which bar recording is used. A claim of absence: reproduced when the bands overlap
under every combination.

In [ ]:
rng = np.random.default_rng(904)
fw = pd.concat([judge(cells[(cells.reading == rd) & cells.group.isin(GROUPS)], ["fwhm_svd", "fwhm_peak"], rng).assign(recording=rd)
                for rd in ret1.READINGS], ignore_index=True)
print(fw.round(4).to_string(index=False))
print("\nCombinations:", len(fw), "; bands separate in", int(fw.separate.sum()),
      "; of these with mutant at smaller sizes:", int((fw.separate & (fw.mutant_smaller_pts > 0) & (fw.mutant_larger_pts == 0)).sum()))
print("\nSpread (5th to 95th percentile, um), reading A, svd:")
for g in GROUPS:
    v = cells[(cells.reading == "A") & (cells.group == g)].fwhm_svd.dropna()
    print(f"  {g}: {v.quantile(0.05):.1f} to {v.quantile(0.95):.1f}; cells at one bar width or less: {(v <= cells[(cells.reading == 'A') & (cells.group == g)].bar_um.loc[v.index]).sum()} of {len(v)}")
print("\nBy retina (reading A):")
print(by_retina(cells[cells.reading == "A"], ["fwhm_svd", "fwhm_peak", "bar_um"]).to_string())

In [ ]:
rng = np.random.default_rng(9041)
v = variants(rng, ["fwhm_svd", "fwhm_peak"])
print(v.round(4).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
A = cells[cells.reading == "A"]
ecdf_plot(axes[0], A, "fwhm_svd", "Centre FWHM, reading A, svd (binomial band)", "um")
ecdf_plot(axes[1], A[A.peak_ratio > 2], "fwhm_svd", "Same, cells with STA peak ratio above 2", "um")
for ax in axes: ax.set_xlim(0, 600)
plt.tight_layout(); plt.show()
print("Cells beyond 600 um (not shown):", int((A.fwhm_svd > 600).sum()), "; all units:", int(A.fwhm_svd.notna().sum()))

**Result: not reproduced.** In all 18 combinations of recording reading, separation and band, the
95% bands separate with the mutant centres smaller (reading A, svd: median 116 against 156 um, KS
p below 1e-4). Much of the difference comes from mutant cells with weak STAs. Among cells whose STA
peak is more than twice the null's (93 mutant, 107 control) the medians are 143 and 158 um (KS
p = 0.11), the binomial and bootstrap bands separate only between 82 and 101 um, and the DKW bands
overlap. The bands also separate without the duplicate units, without 20080702_R4, with the
out-of-order spikes moved, and with 16.6 um bars only.

## s3-receptive-field-surrounds

> Some control and mutant RGCs have receptive field surrounds of opposite sign to the centre, so
> lateral inhibition functions in mutant circuits.

A lobe of b(x) of opposite sign to the centre on both flanks, deeper than 10% (a) or 20% (b) of
the centre peak. Reproduced when, under every reading, at least one mutant and one control cell
have one. The same rule applied to the null STA (spike counts shifted by half the recording) is
shown as the reference for noise.

In [ ]:
rows = []
for rd in ret1.READINGS:
    d = cells[(cells.reading == rd) & cells.group.isin(GROUPS)]
    for sep in ("svd", "peak"):
        for frac in (10, 20):
            for g in GROUPS:
                x = d[d.group == g]
                strong = x[x.peak_ratio > 2]
                rows.append({"recording": rd, "separation": sep, "depth_pct": frac, "group": g, "cells": len(x),
                             "with_surround": int(x[f"surround_{sep}_{frac}"].sum()),
                             "null_with_surround": int(x[f"null_surround_{sep}_{frac}"].sum()),
                             "ratio_above_2": len(strong), "ratio_above_2_with_surround": int(strong[f"surround_{sep}_{frac}"].sum())})
sur = pd.DataFrame(rows)
print(sur.to_string(index=False))
ok = sur.groupby(["recording", "separation", "depth_pct"]).with_surround.min()
print("\nReadings in which both groups have at least one cell with a surround:", int((ok > 0).sum()), "of", len(ok))
print("By retina (reading A, svd, 20%):")
A = cells[(cells.reading == "A") & cells.group.isin(GROUPS)]
print(A.groupby(["group", "retina"]).agg(cells=("cell", "size"), surround=("surround_svd_20", "sum"), null=("null_surround_svd_20", "sum")).to_string())

**Result: reproduced by the criterion, but the test does not show surrounds.** In all 12
readings at least one mutant and one control cell have a lobe of opposite sign on both flanks
(reading A, svd, 20%: 102 of 200 mutant and 16 of 122 control cells). The same rule applied to
null STAs (spike counts shifted by half the recording) is met by 194 to 198 of 200 mutant and 119
to 121 of 122 control cells. So noise alone meets the rule, and the result is no evidence for
or against surrounds.

## fig9e-slower-time-course

> The time course of the light response is significantly slower in mutant retina, measured as
> the time to peak of the temporal integration function a(t).

Time to peak (s) of a(t), under both separations, both peak readings, the three band readings and
the three recording readings. Reproduced when under every combination the bands separate with the
mutant curve at longer times.

In [ ]:
rng = np.random.default_rng(905)
tcols = [f"ttp_{s}_{p}" for s in ("svd", "peak") for p in ("frame", "parabolic")]
tt = pd.concat([judge(cells[(cells.reading == rd) & cells.group.isin(GROUPS)], tcols, rng).assign(recording=rd)
                for rd in ret1.READINGS], ignore_index=True)
print(tt.round(4).to_string(index=False))
slower = tt.separate & (tt.mutant_larger_pts > 0) & (tt.mutant_smaller_pts == 0)
print("\nCombinations:", len(tt), "; separate with mutant at longer times:", int(slower.sum()),
      "; separate the other way:", int((tt.separate & (tt.mutant_smaller_pts > 0)).sum()), "; overlap:", int((~tt.separate).sum()))
print("Median difference, mutant minus control (ms), range over combinations:",
      f"{(tt.median_mut - tt.median_ctl).min() * 1000:.1f} to {(tt.median_mut - tt.median_ctl).max() * 1000:.1f}; KS p at most {tt.ks_p.max():.2g}")
print("Combinations that also separate the other way:")
print(tt[tt.separate & (tt.mutant_smaller_pts > 0)][["recording", "measure", "ci", "mutant_larger_pts", "mutant_smaller_pts", "sep_range"]].to_string(index=False))
for rd in ret1.READINGS:
    d = cells[(cells.reading == rd) & cells.group.isin(GROUPS)]
    fast = d[d.ttp_peak_frame < 0.03]
    print(f"  reading {rd}: cells with time to peak (peak separation, frame) below 30 ms: "
          f"{fast.groupby('group').size().to_dict()}; their STA peak ratios {np.round(np.sort(fast.peak_ratio.to_numpy()), 2).tolist()}")
print("\nBy retina (reading A):")
print(by_retina(cells[cells.reading == "A"], ["ttp_svd_frame", "ttp_svd_parabolic"]).to_string())

In [ ]:
rng = np.random.default_rng(9051)
v = variants(rng, ["ttp_svd_frame", "ttp_svd_parabolic"])
print(v.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3.5))
A = cells[cells.reading == "A"]
ecdf_plot(ax, A, "ttp_svd_parabolic", "Time to peak of a(t), reading A, svd, parabolic", "ms", scale=1000)
plt.tight_layout(); plt.show()
print("Range shown (ms):", round(A.ttp_svd_parabolic.min() * 1000, 1), "to", round(A.ttp_svd_parabolic.max() * 1000, 1))

**Result: reproduced.** In all 36 combinations the 95% bands separate with the mutant curve at
longer times, and the mutant medians exceed the control medians by 16.1 to 33.3 ms (KS p at most
2e-09). In 8 combinations (peak separation, binomial and bootstrap bands, readings A and C) the
bands also separate the other way, at 1 or 2 values below 30 ms. They come from 3 or 4 mutant cells whose a(t) peaks within 30 ms and whose STA peak is no larger than the null's
(ratios 0.86 to 1.05), cells without a measurable receptive field. The separation at
longer times holds in those combinations too. It holds in every variant, including the cells with
clear STAs (93 mutant, 107 control) and each Cre line alone.

## fig9f-lower-firing-rate

> Mutant RGCs fire at much lower rates than control RGCs in response to flickering gratings.

Spike count over the duration of the flicker stimulus, per cell. Reproduced when the mutant curve
lies at lower rates and the bands separate under every band reading (and here every recording
reading).

In [ ]:
rng = np.random.default_rng(906)
fr = pd.concat([judge(cells[(cells.reading == rd) & cells.group.isin(GROUPS)], ["rate_hz"], rng).assign(recording=rd)
                for rd in ret1.READINGS], ignore_index=True)
fr["median_ratio_mut_ctl"] = fr.median_mut / fr.median_ctl
print(fr.round(4).to_string(index=False))
lower = fr.separate & (fr.mutant_smaller_pts > 0) & (fr.mutant_larger_pts == 0)
print("\nCombinations:", len(fr), "; separate with mutant at lower rates:", int(lower.sum()))
print("\nBy retina (reading A):")
print(by_retina(cells[cells.reading == "A"], ["rate_hz"]).to_string())

In [ ]:
rng = np.random.default_rng(9061)
v = variants(rng, ["rate_hz"])
v["median_ratio_mut_ctl"] = v.median_mut / v.median_ctl
print(v.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(5, 3.5))
A = cells[cells.reading == "A"]
ecdf_plot(ax, A, "rate_hz", "Firing rate during flickering bars, reading A", "Hz")
ax.set_xscale("log"); plt.tight_layout(); plt.show()
print("Range (Hz):", round(A.rate_hz.min(), 3), "to", round(A.rate_hz.max(), 1), "; cells at 0 Hz (not on the log axis):", int((A.rate_hz == 0).sum()))

**Result: reproduced.** In all 9 combinations the bands separate with the mutant curve at lower
rates. Under reading A the mutant median is 3.82 Hz against 9.02 Hz for controls (ratio 0.42;
0.38 to 0.45 over the readings). By retina the mutant medians range from 1.9 to 15.9 Hz and the
control medians from 3.8 to 23.6 Hz. Every variant gives a lower mutant median. With the Pax6-Cre
mutants alone (ratio 0.68), and in the cells with clear STAs (0.65), the DKW bands overlap while
the binomial and bootstrap bands still separate.

## fig9g-threshold-unaltered

> In a Linear-Nonlinear model fitted to the flicker responses, the response threshold of mutant
> RGCs is unaltered.

Theta (in units of the generator's standard deviation), under both fits, both forms, the three
band readings and the three recording readings. A claim of absence: reproduced when the bands
overlap under every combination.

In [ ]:
rng = np.random.default_rng(907)
lcols = [f"theta_{f}_{m}" for f in ("frame", "binned") for m in ("printed", "continuous")]
th = pd.concat([judge(cells[(cells.reading == rd) & cells.group.isin(GROUPS)], lcols, rng).assign(recording=rd)
                for rd in ret1.READINGS], ignore_index=True)
print(th.round(4).to_string(index=False))
print("\nCombinations:", len(th), "; bands separate:", int(th.separate.sum()),
      "; with mutant at higher thresholds:", int((th.separate & (th.mutant_larger_pts > 0) & (th.mutant_smaller_pts == 0)).sum()),
      "; overlap:", int((~th.separate).sum()))
A = cells[(cells.reading == "A") & cells.group.isin(GROUPS)]
print("LN fits with no value (no spikes):", int(A.theta_frame_printed.isna().sum()), "of", len(A))
print("\nBy retina (reading A):")
print(by_retina(cells[cells.reading == "A"], ["theta_frame_continuous", "theta_binned_continuous"]).to_string())

In [ ]:
rng = np.random.default_rng(9071)
v = variants(rng, ["theta_frame_continuous", "theta_binned_printed"])
print(v.round(4).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
A = cells[cells.reading == "A"]
ecdf_plot(axes[0], A, "theta_frame_continuous", "Threshold, frame fit, continuous form", "theta (SD of y)")
ecdf_plot(axes[1], A, "theta_binned_printed", "Threshold, binned fit, printed form", "theta (SD of y)")
plt.tight_layout(); plt.show()
print("Ranges:", A.theta_frame_continuous.agg(["min", "max"]).round(2).tolist(), A.theta_binned_printed.agg(["min", "max"]).round(2).tolist())

**Result: not reproduced.** The paper finds the threshold unaltered. Here the bands separate in
all 36 combinations, with mutant thresholds higher (reading A, frame fit: median 1.12 against
0.44 SD of the generator in the continuous form, 1.69 against 0.01 in the printed form). Part of
this comes from mutant cells with weak STAs, whose generator predicts little of the rate. Among
cells with clear STAs the medians are 0.85 against 0.41 (frame fit, continuous form); the DKW bands
overlap there, while the binomial and bootstrap bands still separate.

## fig9h-gain-reduced

> In the same Linear-Nonlinear model, the gain of mutant RGCs is reduced.

G (Hz per standard deviation of the generator) from the same fits. Reproduced when under every
combination the mutant curve lies at lower gains and the bands separate.

In [ ]:
rng = np.random.default_rng(908)
gcols = [f"gain_{f}_{m}" for f in ("frame", "binned") for m in ("printed", "continuous")]
gn = pd.concat([judge(cells[(cells.reading == rd) & cells.group.isin(GROUPS)], gcols, rng).assign(recording=rd)
                for rd in ret1.READINGS], ignore_index=True)
gn["median_ratio_mut_ctl"] = gn.median_mut / gn.median_ctl
print(gn.round(4).to_string(index=False))
lower = gn.separate & (gn.mutant_smaller_pts > 0) & (gn.mutant_larger_pts == 0)
print("\nCombinations:", len(gn), "; separate with mutant at lower gains:", int(lower.sum()),
      "; separate the other way:", int((gn.separate & (gn.mutant_larger_pts > 0)).sum()), "; overlap:", int((~gn.separate).sum()))
print("By form, combinations with mutant lower and separate:", gn.assign(lower=lower).groupby(gn.measure.str.split("_").str[-1]).lower.agg(["sum", "size"]).to_dict())
print("\nBy retina (reading A):")
print(by_retina(cells[cells.reading == "A"], ["gain_frame_printed", "gain_frame_continuous"]).to_string())

In [ ]:
rng = np.random.default_rng(9081)
v = variants(rng, ["gain_frame_printed", "gain_frame_continuous"])
v["median_ratio_mut_ctl"] = v.median_mut / v.median_ctl
print(v.round(4).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
A = cells[cells.reading == "A"]
ecdf_plot(axes[0], A, "gain_frame_printed", "Gain, frame fit, printed form", "G (Hz per SD of y)")
ecdf_plot(axes[1], A, "gain_frame_continuous", "Gain, frame fit, continuous form", "G (Hz per SD of y)")
for ax in axes: ax.set_xlim(-5, 150)
plt.tight_layout(); plt.show()
print("Ranges:", A.gain_frame_printed.agg(["min", "max"]).round(1).tolist(), A.gain_frame_continuous.agg(["min", "max"]).round(1).tolist(),
      "; cells outside -5 to 150 (printed, continuous):", int(((A.gain_frame_printed < -5) | (A.gain_frame_printed > 150)).sum()),
      int(((A.gain_frame_continuous < -5) | (A.gain_frame_continuous > 150)).sum()))
print(f"Run time of the notebook: {(time.time() - started) / 60:.1f} min")

**Result: blocked (methods_underspecified).** The readings of the nonlinearity's form disagree.
With the printed form (B + G y above theta), mutant gains are lower and the bands separate in all
18 combinations (reading A, frame fit: median 17.9 against 23.1 Hz per SD, ratio 0.77). With the
continuous form (B + G (y - theta)), the bands overlap in 14 of 18 combinations and the medians are
close (29.5 against 30.8, KS p = 0.89). Among cells with clear STAs neither form gives lower mutant
gains (printed 23.1 against 23.4, continuous 33.1 against 31.0).

## Summary

| Claim | Outcome |
|---|---|
| 15 histology claims (Figures 1 to 8, Supplementary Figure 1) | Blocked: data missing |
| fig9a-response-variety, fig9c-on-off-index, fig9-rod-cone, fig9b-direction-selective | Blocked: data missing |
| fig9-cell-counts: 115 mutant and 143 control cells | Not reproduced (200 and 122) |
| fig9d-rf-size-similar: centre size similar | Not reproduced (mutant smaller) |
| s3-receptive-field-surrounds | Reproduced by the criterion; the rule is met by noise |
| fig9e-slower-time-course | Reproduced |
| fig9f-lower-firing-rate | Reproduced |
| fig9g-threshold-unaltered | Not reproduced (mutant higher) |
| fig9h-gain-reduced | Blocked: methods underspecified (the forms of the nonlinearity disagree) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

### How the analysis differs from the paper's

- The groups are the data set's labels. They hold 200 mutant and 122 control units, where the
  paper has 115 and 143. The control retinas are mostly labelled "WT (chx10 het)", a genotype the
  paper does not list among its controls. The files do not say which region of the Pax6-Cre
  retinas was recorded, so the mutant group cannot be restricted to the periphery as the paper's
  was. The Bax -/- retina is left out, as the paper did not use Bax mutants for Figure 9.
- All units are used, as the claims file reads the legend's counts; the paper does not say
  whether it kept every unit. Many mutant units have weak or no receptive fields (STA peak below
  twice that of the null in 107 of 200 mutant units and 15 of 122 control units). Their noisy
  STAs give small centres, a high LN threshold and a low printed-form gain, and they carry much
  of the differences in Figure 9D, 9G and 9H.
- Which bar recording is used is not fixed by the claims file. Three readings were run and all
  agree on every outcome.
- The 95% bands are built in the three ways the claims file names, as the paper does not say how
  its bands were made. The LN fit and the separation of h follow the claims file's readings.
- The mutant retinas include all 8.3 um bar recordings; all control retinas were stimulated with
  16.6 um bars. The results hold with the 16.6 um recordings alone.

### What was found about the data set

- It holds only the white-noise part of Figure 9: none of the histology of Figures 1 to 8, and no
  flashing spots, flashes, moving bars or gratings (Figure 9A to 9C).
- Its unit counts (200 mutant, 122 control) and control genotypes differ from the paper's Figure
  9 (115 and 143). The data set does not say how its units relate to the paper's.
- In the 8 recordings of four retinas (20080628_R4 and R6, 20080702_R1 and R4), 82 of 86 spike
  trains run to 1440 s and then hold a second segment between 720 and 780 s, so they are not
  sorted. This is consistent with their last minute (1440 to 1500 s) stored with a 720 s offset in
  place of 1440 s.
- The stimulus of the vertical-bar recording of 20080702_R4 cannot be rebuilt from its
  description: none of its 12 units has an STA, though all 12 respond to the checkerboard of the
  same retina. Its frame length (16.665 ms) also differs from the others'.
- 16 pairs of units share 70 to 100% of their spikes within 0.5 ms, most likely single neurons
  sorted twice.
- The files do not give the age, the Pcdh-gamma allele, the recorded region of the Pax6-Cre
  retinas, or which units are ganglion cells of which type.
- `ran1.bin` matches the RAN1 generator of `ran1.m` with seed -10000, read least significant bit
  first, and the description's table of retinas, cells and labels matches the files.